# Lesson 3 Assignment – Python Library Management System
**Student:** Meryulis Zambrano | BIA – Willis College

**GitHub Link:** [https://github.com/meryuliszambrano-prog/BIA-Lab3-Python-Refresher](https://github.com/meryuliszambrano-prog/BIA-Lab3-Python-Refresher)

#### **ATTENTION!!!**
- **Place each section (1,2...) in one or more cells to keep the notebook organised**
- **Complete the codes in the `TODO` sections**

In [1]:
# Python Library Management System

# Install required libraries (This should already exist in colab)
!pip install requests

import os
import pickle
import sqlite3
import requests

## 1. Data Types and Variables

In [2]:
book_title = "Python Programming"   # str
book_quantity = 5                   # int
book_price = 29.99                  # float
is_available = True                 # bool

print(f"Book: {book_title}, Quantity: {book_quantity}, Price: ${book_price}, Available: {is_available}")
print(type(book_title), type(book_quantity), type(book_price), type(is_available))

Book: Python Programming, Quantity: 5, Price: $29.99, Available: True
<class 'str'> <class 'int'> <class 'float'> <class 'bool'>


## 2. Lists

In [3]:
books = ["Python Basics", "Advanced Python", "Data Science with Python"]

# Add two more books to the list
books.append("Machine Learning with Python")   # append adds the book at the end
books.insert(1, "Python for Data Analysis")    # insert adds the book at position 1

print("Updated book list:", books)
print("Number of books:", len(books))

Updated book list: ['Python Basics', 'Python for Data Analysis', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python']
Number of books: 5


## 3. Loops and range()

In [4]:
print("\nAvailable books:")
# Use a for loop and range() to print books with their indices
for i in range(len(books)):
    print(f"{i}: {books[i]}")


Available books:
0: Python Basics
1: Python for Data Analysis
2: Advanced Python
3: Data Science with Python
4: Machine Learning with Python


## 4. User Input

In [5]:
new_book = input("Enter a new book title: ").strip()

# Edge cases: empty title or a book that is already in the list
if new_book == "":
    print("No title entered. The list was not changed.")
elif new_book in books:
    print(f"'{new_book}' is already in the library.")
else:
    books.append(new_book)
print("Updated book list:", books)

Enter a new book title: Deep Learning Fundamentals
Updated book list: ['Python Basics', 'Python for Data Analysis', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Fundamentals']


## 5. Functions and Conditional Statements

In [6]:
def check_availability(book_name):
    """Return True if the book is in the list, False otherwise (not case sensitive)."""
    if not isinstance(book_name, str) or book_name.strip() == "":
        return False
    if book_name.strip().lower() in [b.lower() for b in books]:
        return True
    else:
        return False

# Test the function
print(f"Is 'Python Basics' available? {check_availability('Python Basics')}")
print(f"Is 'python basics' available? {check_availability('python basics')}")
print(f"Is 'Java for Beginners' available? {check_availability('Java for Beginners')}")
print(f"Is '' available? {check_availability('')}")

Is 'Python Basics' available? True
Is 'python basics' available? True
Is 'Java for Beginners' available? False
Is '' available? False


## 6. File I/O

In [7]:
def save_books_to_file(filename):
    with open(filename, 'w') as f:
        for book in books:
            f.write(f"{book}\n")

def read_books_from_file(filename):
    """Read the books from a text file (one title per line) and return a list."""
    try:
        with open(filename, 'r') as f:
            return [line.strip() for line in f if line.strip() != ""]
    except FileNotFoundError:
        print(f"Error: the file '{filename}' does not exist.")
        return []

# Test file I/O
save_books_to_file("books.txt")
!cat books.txt  # Display file contents in Colab

print("Books read from file:", read_books_from_file("books.txt"))
print("Missing file test:", read_books_from_file("no_file.txt"))

Python Basics
Python for Data Analysis
Advanced Python
Data Science with Python
Machine Learning with Python
Deep Learning Fundamentals


Books read from file: ['Python Basics', 'Python for Data Analysis', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Fundamentals']
Error: the file 'no_file.txt' does not exist.
Missing file test: []


## 7. Pickle for serialization

In [8]:
def save_books_pickle(filename):
    with open(filename, 'wb') as f:
        pickle.dump(books, f)

def load_books_pickle(filename):
    """Load the list of books from a pickle file."""
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        print(f"Error: the file '{filename}' does not exist.")
        return []
    except pickle.UnpicklingError:
        print(f"Error: '{filename}' is not a valid pickle file.")
        return []

# Test pickle functions
save_books_pickle("books.pkl")
print("Books loaded from pickle:", load_books_pickle("books.pkl"))
print("Missing file test:", load_books_pickle("missing.pkl"))

Books loaded from pickle: ['Python Basics', 'Python for Data Analysis', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Fundamentals']
Error: the file 'missing.pkl' does not exist.
Missing file test: []


## 8. SQLite Database Interaction

In [9]:
def create_books_table():
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS books
                      (id INTEGER PRIMARY KEY, title TEXT, quantity INTEGER)''')
    conn.commit()
    conn.close()

def add_book_to_db(title, quantity):
    """Insert a book in the database. If the title exists, add to its quantity."""
    if not title or quantity < 0:
        print("Error: title is required and quantity cannot be negative.")
        return
    conn = None
    try:
        conn = sqlite3.connect('library.db')
        cursor = conn.cursor()
        cursor.execute("SELECT id, quantity FROM books WHERE title = ?", (title,))
        row = cursor.fetchone()
        if row:
            cursor.execute("UPDATE books SET quantity = ? WHERE id = ?", (row[1] + quantity, row[0]))
            print(f"'{title}' already exists. New quantity: {row[1] + quantity}")
        else:
            cursor.execute("INSERT INTO books (title, quantity) VALUES (?, ?)", (title, quantity))
            print(f"'{title}' added with quantity {quantity}")
        conn.commit()
    except sqlite3.Error as e:
        print("Database error:", e)
    finally:
        if conn:
            conn.close()

def show_books_in_db():
    """Print all the books stored in the database."""
    conn = sqlite3.connect('library.db')
    rows = conn.execute("SELECT id, title, quantity FROM books ORDER BY id").fetchall()
    conn.close()
    for row in rows:
        print(row)

# Test database functions
if os.path.exists('library.db'):
    os.remove('library.db')       # start with an empty database every run
create_books_table()
add_book_to_db("Python for Beginners", 10)
add_book_to_db("Data Science with Python", 4)
add_book_to_db("Python for Beginners", 2)     # existing title: quantity is updated
add_book_to_db("", 3)                         # edge case: empty title
add_book_to_db("Bad Quantity Book", -1)       # edge case: negative quantity
show_books_in_db()

'Python for Beginners' added with quantity 10
'Data Science with Python' added with quantity 4
'Python for Beginners' already exists. New quantity: 12
Error: title is required and quantity cannot be negative.
Error: title is required and quantity cannot be negative.
(1, 'Python for Beginners', 12)
(2, 'Data Science with Python', 4)


## 9. Web API Interaction (GitHub API)

In [10]:
def get_python_repos():
    url = "https://api.github.com/search/repositories"
    params = {"q": "language:python", "sort": "stars", "order": "desc"}
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()           # error if the status is not 200
        data = response.json()
        return [repo['name'] for repo in data['items'][:5]]
    except requests.exceptions.RequestException as e:
        print("Error connecting to the GitHub API:", e)
        return []

# Test API function
print("Top 5 Python repositories on GitHub:", get_python_repos())

Error connecting to the GitHub API: 403 Client Error: Forbidden for url: https://api.github.com/search/repositories?q=language%3Apython&sort=stars&order=desc
Top 5 Python repositories on GitHub: []


## 10. Extra features: remove and update books

In [11]:
def remove_book(book_name):
    """Remove a book from the list if it exists."""
    if book_name in books:
        books.remove(book_name)
        print(f"'{book_name}' was removed.")
    else:
        print(f"'{book_name}' is not in the library.")

def update_book(old_name, new_name):
    """Change the title of a book in the list."""
    if old_name in books and new_name.strip() != "":
        books[books.index(old_name)] = new_name.strip()
        print(f"'{old_name}' was renamed to '{new_name.strip()}'.")
    else:
        print("Update not possible: check the titles.")

def update_quantity_db(title, quantity):
    """Update the quantity of a book in the database."""
    conn = sqlite3.connect('library.db')
    cursor = conn.execute("UPDATE books SET quantity = ? WHERE title = ?", (quantity, title))
    conn.commit()
    conn.close()
    print("Rows updated:", cursor.rowcount)

def remove_book_db(title):
    """Delete a book from the database."""
    conn = sqlite3.connect('library.db')
    cursor = conn.execute("DELETE FROM books WHERE title = ?", (title,))
    conn.commit()
    conn.close()
    print("Rows deleted:", cursor.rowcount)

# Tests
remove_book("Advanced Python")
remove_book("Java for Beginners")
update_book("Python Basics", "Python Basics - 2nd Edition")
print("Books:", books)

update_quantity_db("Data Science with Python", 7)
remove_book_db("Python for Beginners")
show_books_in_db()

'Advanced Python' was removed.
'Java for Beginners' is not in the library.
'Python Basics' was renamed to 'Python Basics - 2nd Edition'.
Books: ['Python Basics - 2nd Edition', 'Python for Data Analysis', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Fundamentals']
Rows updated: 1
Rows deleted: 1
(2, 'Data Science with Python', 7)


## 11. Simple command-line interface

In [12]:
def library_menu():
    """Simple menu to use the library from the keyboard."""
    while True:
        print("\n--- Library Menu ---")
        print("1. Show books  2. Add book  3. Check book  4. Remove book  5. Exit")
        option = input("Choose an option: ").strip()
        if option == "1":
            for i in range(len(books)):
                print(f"{i}: {books[i]}")
        elif option == "2":
            title = input("Title: ").strip()
            if title and title not in books:
                books.append(title)
                print(f"'{title}' added.")
            else:
                print("Empty or duplicate title.")
        elif option == "3":
            title = input("Title: ")
            print("Available" if check_availability(title) else "Not available")
        elif option == "4":
            remove_book(input("Title: ").strip())
        elif option == "5":
            print("Goodbye!")
            break
        else:
            print("Invalid option, try again.")

library_menu()


--- Library Menu ---
1. Show books  2. Add book  3. Check book  4. Remove book  5. Exit
Choose an option: 1
0: Python Basics - 2nd Edition
1: Python for Data Analysis
2: Data Science with Python
3: Machine Learning with Python
4: Deep Learning Fundamentals

--- Library Menu ---
1. Show books  2. Add book  3. Check book  4. Remove book  5. Exit
Choose an option: 2
Title: Statistics for Data Analysts
'Statistics for Data Analysts' added.

--- Library Menu ---
1. Show books  2. Add book  3. Check book  4. Remove book  5. Exit
Choose an option: 3
Title: python basics - 2nd edition
Available

--- Library Menu ---
1. Show books  2. Add book  3. Check book  4. Remove book  5. Exit
Choose an option: 4
Title: Machine Learning with Python
'Machine Learning with Python' was removed.

--- Library Menu ---
1. Show books  2. Add book  3. Check book  4. Remove book  5. Exit
Choose an option: 7
Invalid option, try again.

--- Library Menu ---
1. Show books  2. Add book  3. Check book  4. Remove book 

In [13]:
# Save the final list
save_books_to_file("books.txt")
save_books_pickle("books.pkl")
print("Final list from pickle:", load_books_pickle("books.pkl"))

print("\nLibrary Management System operations completed.")

Final list from pickle: ['Python Basics - 2nd Edition', 'Python for Data Analysis', 'Data Science with Python', 'Deep Learning Fundamentals', 'Statistics for Data Analysts']

Library Management System operations completed.
